In [ ]:
# Carregamento dos dados
import duckdb
import pandas as pd
import psutil
import time

arquivo = "yellow_tripdata_2026-01.parquet"

#  0   VendorID               int32         
#  1   tpep_pickup_datetime   datetime64[us]
#  2   tpep_dropoff_datetime  datetime64[us]
#  3   passenger_count        Int64         
#  4   trip_distance          float64       
#  5   RatecodeID             Int64         
#  6   store_and_fwd_flag     str           
#  7   PULocationID           int32         
#  8   DOLocationID           int32         
#  9   payment_type           int64         
#  10  fare_amount            float64       
#  11  extra                  float64       
#  12  mta_tax                float64       
#  13  tip_amount             float64       
#  14  tolls_amount           float64       
#  15  improvement_surcharge  float64       
#  16  total_amount           float64       
#  17  congestion_surcharge   float64       
#  18  Airport_fee            float64       
#  19  cbd_congestion_fee     float64 

In [ ]:
# Inspecionar o número de linhas, as colunas e os tipos de dados
df = pd.read_parquet(arquivo)

con = duckdb.connect()
con.execute(f"""
   CREATE TABLE corridas AS
   SELECT * FROM read_parquet('{arquivo}')
""")

con.table("corridas").show()

df = con.table("corridas").df()
df.info()

### Questão 1

**Corridas por horário (sem IA):** Para cada hora de início da corrida (0 a 23, a partir de tpep_pickup_datetime), calcule o número de corridas, a distância média (trip_distance) e o valor médio pago (total_amount). Ordene pelo número de corridas, do maior para o menor. Utilize os registros de janeiro de 2026.

In [ ]:
# DuckDB

df_duckdb_hora_inicio = con.execute(f"""
    SELECT 
        HOUR(tpep_pickup_datetime) AS hora_inicio_corrida,
        COUNT(*) AS num_corridas,
        AVG(trip_distance) AS distancia_media,
        AVG(total_amount) AS valor_medio_pago
    FROM read_parquet('{arquivo}')
    GROUP BY hora_inicio_corrida
    ORDER BY num_corridas DESC
""").df()

print(df_duckdb_hora_inicio)

In [ ]:
# Pandas

df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])

df_pandas_hora_inicio = (
    df.groupby(df['tpep_pickup_datetime'].dt.hour).agg(
                                                        num_corridas=('tpep_pickup_datetime', 'count'),
                                                        distancia_media=('trip_distance', 'mean'),
                                                        valor_medio_pago=('total_amount', 'mean'),
                                                    ).reset_index(names='hora_inicio_corrida').sort_values('num_corridas', ascending=False)
)

print(df_pandas_hora_inicio)

### Questão 2

**Corridas semelhantes (com IA)**: Encontre pares de corridas distintas que tenham a mesma origem (PULocationID), o mesmo destino (DOLocationID) e horários de início (tpep_pickup_datetime) com diferença de, no máximo, cinco minutos. Conte cada par apenas uma vez e apresente somente a quantidade total de pares. Para limitar o volume, considere as corridas iniciadas em 15/01/2026, das 8h às 9h(sem incluir 9h).

In [ ]:
import pandas as pd

# Janela de tempo (início inclusive, fim exclusivo)
inicio = "2026-01-15 08:00:00"
fim = "2026-01-15 09:00:00"


# ---------------------------------------------------------
# Implementação com Pandas
# ---------------------------------------------------------
def pares_pandas(df):
    # 1) Filtra as corridas da janela de tempo
    mascara = (df["tpep_pickup_datetime"] >= inicio) & (df["tpep_pickup_datetime"] < fim)
    janela = df.loc[mascara, ["PULocationID", "DOLocationID", "tpep_pickup_datetime"]]

    # 2) Guarda o índice original para identificar cada corrida
    janela = janela.reset_index().rename(columns={"index": "id"})

    # 3) Auto-junção: mesma origem e mesmo destino
    juntas = janela.merge(
        janela,
        on=["PULocationID", "DOLocationID"],
        suffixes=("_a", "_b"),
    )

    # 4) id_a < id_b: corridas distintas e cada par contado uma única vez
    juntas = juntas[juntas["id_a"] < juntas["id_b"]]

    # 5) Diferença de horário de no máximo 5 minutos
    diferenca = (juntas["tpep_pickup_datetime_a"] - juntas["tpep_pickup_datetime_b"]).abs()
    juntas = juntas[diferenca <= pd.Timedelta(minutes=5)]

    return len(juntas)

In [ ]:
import duckdb

# ---------------------------------------------------------
# Implementação com DuckDB
# ---------------------------------------------------------
def pares_duckdb(con):
    consulta = f"""
        WITH janela AS (
            SELECT
                rowid AS id,
                PULocationID,
                DOLocationID,
                tpep_pickup_datetime
            FROM corridas
            WHERE tpep_pickup_datetime >= TIMESTAMP '{inicio}'
              AND tpep_pickup_datetime <  TIMESTAMP '{fim}'
        )
        SELECT COUNT(*) AS total_pares
        FROM janela a
        JOIN janela b
          ON  a.PULocationID = b.PULocationID
          AND a.DOLocationID = b.DOLocationID
          AND a.id < b.id
          AND b.tpep_pickup_datetime BETWEEN a.tpep_pickup_datetime - INTERVAL 5 MINUTE
                                         AND a.tpep_pickup_datetime + INTERVAL 5 MINUTE
    """
    return con.execute(consulta).fetchone()[0]

In [ ]:
# ---------------------------------------------------------
# Execução e comparação
# ---------------------------------------------------------
total_pandas = pares_pandas(df)
total_duckdb = pares_duckdb(con)

print("Pandas:", total_pandas)
print("DuckDB:", total_duckdb)

assert total_pandas == total_duckdb, "Os resultados são diferentes!"

<!-- Sugestões para as perguntas 3 e 4:

* Quais são os X lugares de embarque com maior número de corridas? (usar coluna PULocationID)
* Quais são os X destinos com maior valor médio por corrida? ()
* Como a duração média das corridas varia ao longo do dia? ps: IA
* Quais locais de origem E destino são mais frequentes em corridas? ps: IA de novo -->
### Questão 3

**Consulta livre (sem IA)**: Quais são os destinos com maior valor médio e distância média por corrida?

In [ ]:
# 11:08 - 11:42
 
t_load_avg = 0
t_exec_avg = 0

df_duckdb_media_destinos = pd.DataFrame()

for i in range (1, 4):
    # load  
    t_load_s = time.time()

    con = duckdb.connect()
    con.execute(f"""
    CREATE TABLE corridas AS
    SELECT * FROM read_parquet('{arquivo}')
    """)

    t_load = time.time() - t_load_s
    t_load_avg += t_load

    print(f"Run {i}: loaded data in {round(t_load, 2)}s")

    # exec
    t_exec_s = time.time()

    # select dolocationid, avg(fare_amount) order by avg DESC
    df_duckdb_media_destinos = con.execute(f"""
        SELECT 
            DOLocationID AS locationID,
            AVG(total_amount - tip_amount) AS avg_price,
            AVG(trip_distance) AS avg_distance
        FROM corridas
        WHERE 
            total_amount - tip_amount > 0
            AND trip_distance > 0
        GROUP BY locationID
        ORDER BY avg_price DESC, avg_distance DESC
    """).df()

    t_exec = time.time() - t_exec_s
    t_exec_avg += t_exec

    print(f"Run {i}: done in {round(t_exec, 2)}s")
    
print(f"\nAverage load time: {round(t_load_avg/3, 2)}s")
print(f"Average exec time: {round(t_exec_avg/3, 2)}s")
print(df_duckdb_media_destinos)

### Questão 4

**Consulta livre (com IA)**: Como a duração média das corridas varia ao longo do dia? 

In [ ]:
t_load_avg = 0
t_exec_avg = 0

resultado_duckdb = pd.DataFrame()

for i in range (1, 4):
    # load  
    t_load_s = time.time()

    con = duckdb.connect()
    con.execute(f"""
    CREATE TABLE corridas AS
    SELECT * FROM read_parquet('{arquivo}')
    """)

    t_load = time.time() - t_load_s
    t_load_avg += t_load

    print(f"Run {i}: loaded data in {round(t_load, 2)}s")

    # exec
    t_exec_s = time.time()

    resultado_duckdb = con.execute("""
        SELECT
            EXTRACT(hour FROM tpep_pickup_datetime)::INT AS hora,
            ROUND(AVG(
                date_diff('microsecond', tpep_pickup_datetime, tpep_dropoff_datetime)
                / 60000000.0
            ), 2) AS duracao_media_min,
            COUNT(*) AS total_corridas
        FROM corridas
        WHERE tpep_dropoff_datetime > tpep_pickup_datetime
        GROUP BY hora
        ORDER BY hora
    """).df()

    t_exec = time.time() - t_exec_s
    t_exec_avg += t_exec

    print(f"Run {i}: done in {round(t_exec, 2)}s")
    
print(f"\nAverage load time: {round(t_load_avg/3, 2)}s")
print(f"Average exec time: {round(t_exec_avg/3, 2)}s")
print(resultado_duckdb)

In [ ]:
# Alinha os tipos antes de comparar
# resultado_pandas["hora"] = resultado_pandas["hora"].astype("int64")
# resultado_duckdb["hora"] = resultado_duckdb["hora"].astype("int64")
# # resultado_pandas["total_corridas"] = resultado_pandas["total_corridas"].astype("int64")
# resultado_duckdb["total_corridas"] = resultado_duckdb["total_corridas"].astype("int64")

# pd.testing.assert_frame_equal(
#     # resultado_pandas,
#     resultado_duckdb,
#     check_dtype=False,
#     atol=0.01,
# )
# print("Os dois resultados são equivalentes.")

### Questão 5

**Corridas simultâneas (sem IA)**: Considere as corridas iniciadas no dia 15/01/2026 a partir das 8h. Conte quantos pares de corridas tiveram seus períodos de viagem sobrepostos, considerando os horários de início (tpep_pickup_datetime) e de término (tpep_dropoff_datetime). Considere apenas pares de corridas distintas e conte cada par uma única vez. Execute a consulta para três janelas de tempo, das 8h às 9h, das 8h às 10h e das 8h às 12h (sem incluir o horário final). Informe a quantidade de pares e o tempo de execução de cada janela. Se alguma execução exceder a memória disponível, registre isso e não aumente a janela.

In [ ]:
time_windows = [
    ("08:00:00", "09:00:00"),
    ("08:00:00", "10:00:00"),
    ("08:00:00", "12:00:00")
]

In [ ]:
# 00:18 - 00:40: idealização da implementação
# 13:21 - 14:27: cerca de 20 minutos lendo documentação + stack overflow (como encontrei rowid)
# 14:47 - 15:14: finalização da query do duckdb e adicionar time tracking simples

# DuckDB
t_load_avg = 0

for i in range (1, 4):
    # load  
    t_load_s = time.time()

    con = duckdb.connect()
    con.execute(f"""
    CREATE TABLE corridas AS
    SELECT * FROM read_parquet('{arquivo}')
    """)

    t_load = time.time() - t_load_s
    t_load_avg += t_load

    print(f"Run {i}: loaded data in {round(t_load, 2)}s")

    # exec
    t_exec_avg = 0

    for t_start, t_end in time_windows:
        t_exec_s = time.time()
    
        try:
            df_duckdb_periodo_viagem = con.execute(f"""
                SELECT 
                    COUNT(*) AS unique_pairs_count
                FROM (SELECT 
                        c1.rowid AS tpep1,
                        c2.rowid AS tpep2           
                    FROM corridas c1, corridas c2
                    WHERE 
                        c1.tpep_pickup_datetime >= '2026-01-15 {t_start}'
                        AND c2.tpep_pickup_datetime >= '2026-01-15 {t_start}'
                        AND c1.tpep_pickup_datetime < '2026-01-15 {t_end}'
                        AND c2.tpep_pickup_datetime < '2026-01-15 {t_end}'
                        AND c1.tpep_pickup_datetime < c2.tpep_dropoff_datetime
                        AND c2.tpep_pickup_datetime < c1.tpep_dropoff_datetime)
                WHERE tpep1 < tpep2 
            """).df()
        except:
            print(f"Run {i}: {t_start} to {t_end} exceeded available memory")
            break

        t_exec = time.time() - t_exec_s
        t_exec_avg += t_exec

        print(f"Run {i}: {t_start} to {t_end} - {df_duckdb_periodo_viagem['unique_pairs_count'][0]:8d} unique pairs in {t_exec}s")
    
    print(f"Run {i}: t_exec_avg = {round(t_exec_avg/3, 2)}s\n")

print(f"Average load time: {round(t_load_avg/3, 2)}s")

In [ ]:
# 15:41 - 16:15: cerca de 10 minutos lendo documentação do pd.read_parquet() df.merge(); resto fazendo query
# 16:20 - 17:04: cerca de 15 minutos lendo documentação do psutil + stack overflow; problemas com memória, tentando impedir que o vscode crashe :/
#Pandas
BYTES_ROW = 6 * 8 # 6 columns * 8 bytes (int64s and datetime64s)

memory_exceeded = False

num_loads = 0
t_load_avg = 0

for i in range (1, 4):
    # load
    t_load_s = time.time()

    df = pd.read_parquet(arquivo, columns=["tpep_pickup_datetime", "tpep_dropoff_datetime"])

    t_load = time.time() - t_load_s
    t_load_avg += t_load
    num_loads += 1

    print(f"Run {i}: loaded data in {round(t_load, 2)}s")

    # exec
    t_exec_avg = 0
    num_execs = 0

    for t_start, t_end in time_windows:
        t_exec_s = time.time()

        df_tpep = df.query(
            f"tpep_pickup_datetime >= '2026-01-15 {t_start}' "
            f"and tpep_pickup_datetime < '2026-01-15 {t_end}'"
        ).reset_index(names="tpep_id")

        df_tpep_len = len(df_tpep)

        # est = df_tpep_len ** 2 * BYTES_ROW
        # avail = psutil.virtual_memory().available
        # print(f"n = {df_tpep_len:,}  estimated = {est/1e9:.1f}GB  available = {avail/1e9:.1f}GB")

        if df_tpep_len * df_tpep_len * BYTES_ROW > psutil.virtual_memory().available:
            memory_exceeded = True
        else:
            try:
                df_pairs = df_tpep.merge(df_tpep, how="cross", suffixes=("_1", "_2")).query(
                    "tpep_id_1 < tpep_id_2 "
                    "and tpep_pickup_datetime_1 < tpep_dropoff_datetime_2 "
                    "and tpep_pickup_datetime_2 < tpep_dropoff_datetime_1"
                )

                pairs_count = len(df_pairs)
                del df_pairs
            except:
                memory_exceeded = True

        if memory_exceeded:
            print(f"Run {i}: {t_start} to {t_end} exceeded available memory")
            break

        t_exec = time.time() - t_exec_s
        t_exec_avg += t_exec
        num_execs += 1

        print(f"Run {i}: {t_start} to {t_end} - {pairs_count:8d} unique pairs in {round(t_exec, 2)}s")

    if memory_exceeded:
        print(f"Further windows will not be considered; stopped at run {i}\n")
        break

    print(f"Run {i}: t_exec_avg = {round(t_exec_avg / num_execs, 2)}s\n")

print(f"Average load time: {round(t_load_avg / num_loads, 2)}s")